In [0]:
# Demo 15 Setup: Publishing a Dashboard
# Creates Gold-layer tables AND a draft dashboard for students to practice publishing.

import re
import json
import requests
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from datetime import date, timedelta
import random

# --- Schema setup ---
username = spark.sql("SELECT current_user()").collect()[0][0]
clean_username = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])
catalog = "main"
schema = f"demo_publishing_{clean_username}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

print(f"Schema: {catalog}.{schema}")

In [0]:
# --- gold_sales: 2000 rows ---
random.seed(42)

spark.sql("DROP TABLE IF EXISTS gold_sales")

regions = ['Northeast', 'Southeast', 'Midwest', 'West', 'Northwest']
categories = ['Electronics', 'Clothing', 'Home & Garden', 'Sports', 'Food & Beverage']
segments = ['Enterprise', 'Mid-Market', 'SMB', 'Consumer']
channels = ['Online', 'Retail Store', 'Partner', 'Mobile App']

schema_def = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("order_date", DateType(), False),
    StructField("region", StringType(), False),
    StructField("product_category", StringType(), False),
    StructField("customer_segment", StringType(), False),
    StructField("channel", StringType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("net_revenue", DoubleType(), False),
    StructField("cost", DoubleType(), False)
])

rows = []
for i in range(1, 2001):
    order_date = date(2024, 1, 1) + timedelta(days=random.randint(0, 364))
    unit_price = round(random.uniform(15, 500), 2)
    qty = random.randint(1, 6)
    discount = random.choice([0.0, 0.0, 0.0, 0.05, 0.10, 0.15, 0.20])
    net_rev = round(qty * unit_price * (1 - discount), 2)
    cost_val = round(net_rev * random.uniform(0.4, 0.7), 2)
    rows.append((
        i, order_date,
        random.choice(regions), random.choice(categories),
        random.choice(segments), random.choice(channels),
        qty, unit_price, net_rev, cost_val
    ))

df = spark.createDataFrame(rows, schema=schema_def)
df.write.saveAsTable("gold_sales")
print(f"Created gold_sales: {df.count()} rows")

In [0]:
# --- Create a draft dashboard programmatically ---
# This uses the Lakeview API to create a dashboard with datasets and widgets.
# Students will practice PUBLISHING this dashboard in the demo.

import urllib.parse

# Get workspace context
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
parent_path = f"/Users/{username}"

# Fully qualified table reference
table_ref = f"`{catalog}`.`{schema}`.`gold_sales`"

# Define datasets (the SQL queries that power the dashboard)
datasets = [
    {
        "name": "ds_kpi_summary",
        "displayName": "KPI Summary",
        "query": f"SELECT\n  ROUND(SUM(net_revenue), 0) AS total_revenue,\n  COUNT(order_id) AS total_orders,\n  ROUND(AVG(net_revenue), 2) AS avg_order_value,\n  ROUND(SUM(net_revenue) - SUM(cost), 0) AS total_profit\nFROM {table_ref}"
    },
    {
        "name": "ds_revenue_by_region",
        "displayName": "Revenue by Region",
        "query": f"SELECT\n  region,\n  ROUND(SUM(net_revenue), 2) AS total_revenue,\n  COUNT(order_id) AS order_count\nFROM {table_ref}\nGROUP BY region\nORDER BY total_revenue DESC"
    },
    {
        "name": "ds_monthly_trend",
        "displayName": "Monthly Revenue Trend",
        "query": f"SELECT\n  DATE_TRUNC('month', order_date) AS order_month,\n  ROUND(SUM(net_revenue), 2) AS total_revenue,\n  COUNT(order_id) AS order_count\nFROM {table_ref}\nGROUP BY DATE_TRUNC('month', order_date)\nORDER BY order_month"
    },
    {
        "name": "ds_category_breakdown",
        "displayName": "Revenue by Category",
        "query": f"SELECT\n  product_category,\n  ROUND(SUM(net_revenue), 2) AS total_revenue,\n  COUNT(order_id) AS order_count,\n  ROUND((SUM(net_revenue) - SUM(cost)) / SUM(net_revenue) * 100, 1) AS margin_pct\nFROM {table_ref}\nGROUP BY product_category\nORDER BY total_revenue DESC"
    }
]

# Define page layout with widgets
page = {
    "name": "page_main",
    "displayName": "Sales Overview",
    "layout": [
        {
            "widget": {
                "name": "w_title",
                "textbox_spec": "# Sales Performance Dashboard\n\nThis dashboard shows key sales metrics for 2024. Use it to practice **publishing** with different credential modes."
            },
            "position": {"x": 0, "y": 0, "width": 6, "height": 1}
        },
        {
            "widget": {
                "name": "w_revenue_counter",
                "queries": [{"name": "main_query", "query": {"datasetName": "ds_kpi_summary", "fields": [{"name": "total_revenue", "expression": "`total_revenue`"}], "disaggregated": True}}],
                "spec": {
                    "version": 3,
                    "widgetType": "counter",
                    "encodings": {"value": {"fieldName": "total_revenue", "displayName": "Total Revenue"}}
                }
            },
            "position": {"x": 0, "y": 1, "width": 2, "height": 2}
        },
        {
            "widget": {
                "name": "w_orders_counter",
                "queries": [{"name": "main_query", "query": {"datasetName": "ds_kpi_summary", "fields": [{"name": "total_orders", "expression": "`total_orders`"}], "disaggregated": True}}],
                "spec": {
                    "version": 3,
                    "widgetType": "counter",
                    "encodings": {"value": {"fieldName": "total_orders", "displayName": "Total Orders"}}
                }
            },
            "position": {"x": 2, "y": 1, "width": 2, "height": 2}
        },
        {
            "widget": {
                "name": "w_profit_counter",
                "queries": [{"name": "main_query", "query": {"datasetName": "ds_kpi_summary", "fields": [{"name": "total_profit", "expression": "`total_profit`"}], "disaggregated": True}}],
                "spec": {
                    "version": 3,
                    "widgetType": "counter",
                    "encodings": {"value": {"fieldName": "total_profit", "displayName": "Total Profit"}}
                }
            },
            "position": {"x": 4, "y": 1, "width": 2, "height": 2}
        },
        {
            "widget": {
                "name": "w_revenue_by_region",
                "queries": [{"name": "main_query", "query": {"datasetName": "ds_revenue_by_region", "fields": [{"name": "region", "expression": "`region`"}, {"name": "total_revenue", "expression": "`total_revenue`"}], "disaggregated": True}}],
                "spec": {
                    "version": 3,
                    "widgetType": "bar",
                    "encodings": {
                        "x": {"fieldName": "region", "scale": {"type": "categorical"}, "displayName": "Region"},
                        "y": {"fieldName": "total_revenue", "scale": {"type": "quantitative"}, "displayName": "Total Revenue"}
                    }
                }
            },
            "position": {"x": 0, "y": 3, "width": 3, "height": 3}
        },
        {
            "widget": {
                "name": "w_monthly_trend",
                "queries": [{"name": "main_query", "query": {"datasetName": "ds_monthly_trend", "fields": [{"name": "order_month", "expression": "`order_month`"}, {"name": "total_revenue", "expression": "`total_revenue`"}], "disaggregated": True}}],
                "spec": {
                    "version": 3,
                    "widgetType": "line",
                    "encodings": {
                        "x": {"fieldName": "order_month", "scale": {"type": "temporal"}, "displayName": "Month"},
                        "y": {"fieldName": "total_revenue", "scale": {"type": "quantitative"}, "displayName": "Revenue"}
                    }
                }
            },
            "position": {"x": 3, "y": 3, "width": 3, "height": 3}
        }
    ]
}

# Build the serialized dashboard JSON
dashboard_definition = {
    "pages": [page],
    "datasets": datasets
}

serialized = json.dumps(dashboard_definition)

# Create the dashboard via Lakeview API
dashboard_name = f"Sales Dashboard - Publishing Demo ({clean_username})"

response = requests.post(
    f"https://{workspace_url}/api/2.0/lakeview/dashboards",
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json={
        "display_name": dashboard_name,
        "serialized_dashboard": serialized,
        "parent_path": parent_path
    }
)

if response.status_code == 200:
    dashboard_info = response.json()
    dashboard_id = dashboard_info["dashboard_id"]
    dashboard_path = dashboard_info.get("path", f"{parent_path}/{dashboard_name}.lvdash.json")
    print(f"\n✅ Dashboard created successfully!")
    print(f"   Name: {dashboard_name}")
    print(f"   ID:   {dashboard_id}")
    print(f"   Path: {dashboard_path}")
else:
    print(f"\n⚠️ Dashboard creation returned status {response.status_code}")
    print(f"   Response: {response.text}")
    print(f"   This may mean the dashboard already exists. Check your workspace.")
    dashboard_id = "CHECK_WORKSPACE"

In [0]:
# --- Summary ---
print("="*60)
print("SETUP COMPLETE")
print("="*60)
print(f"Catalog:  {catalog}")
print(f"Schema:   {schema}")
print(f"")
print(f"Table created:")
print(f"  gold_sales - 2000 rows")
print(f"")
print(f"Dashboard created:")
print(f"  Name: {dashboard_name}")
print(f"  Status: DRAFT (not yet published)")
print(f"  Contents: 4 datasets, 6 widgets (3 counters + bar + line + text)")
print(f"")
print(f"Your task in the demo: Practice PUBLISHING this dashboard")
print(f"with different credential modes.")
print(f"")
print(f"To find your dashboard:")
print(f"  1. Click 'Dashboards' in the left sidebar")
print(f"  2. Look for: {dashboard_name}")
print(f"  3. Or navigate to: {parent_path}")